## Sequential training. 

This script takes the  the original annotation (for step 1) and the corrected additional labels and trains the models again. It can be re run multiple times by changig the "sequence" argument below. 

In [ ]:
sequence = 7
from opensoundscape.annotations import BoxedAnnotations
from opensoundscape import Audio, audio

import numpy as np
import pandas as pd
from pathlib import Path



In [2]:
# load the training set for this step, it already has the og annotations + the ones from the inference steps
from Keys import audio_file_file_tr_og

step = Path(f"annotation_step{sequence}")
training_set = BoxedAnnotations.from_csv(step / f"training_set_step{sequence}.csv")
# the list of all the files (also the ones without airguns), from_csv does not keep it
files = pd.read_csv(step / f"training_set_step{sequence}_audio_files.csv")["audio_file"].tolist()

Create the clip labels for training from the annotations object (they are not boxes but still)

In [3]:
# Parameters to use for label creation
clip_duration = 5
clip_overlap = 0
min_label_overlap = 0.25

# Create dataframe of one-hot labels
labels = training_set.clip_labels(
    clip_duration = clip_duration,
    clip_overlap = clip_overlap,
    min_label_overlap = min_label_overlap,
    audio_files = files,
)

In [4]:
# split by file so the same recording is not in train and val, the long og file always stays in train
import random
random.seed(3101992)
other_files = [f for f in files if f != str(audio_file_file_tr_og)]
random.shuffle(other_files)
val_files = set(other_files[: int(0.3 * len(other_files))])

is_val = labels.index.get_level_values("file").isin(val_files)
train_df = labels[~is_val]
val_df   = labels[is_val]
print(len(labels))

for name, d in [("train", train_df), ("val", val_df),]:
    print(name, len(d), "clips,",d["airgun"].sum(), "with airgun")

2409
train 1779 clips, 514 with airgun
val 630 clips, 133 with airgun


Load the models and train the resnet and the efficient net

In [5]:
# Create a CNN object designed to recognize 3-second samples
from opensoundscape import CNN

# Use resnet34 architecture
resnet = "resnet18"
effnet = "efficientnet_b0"

# Can use this code to get your classes, if needed
class_list = list(train_df.columns)
class_list
model1 = CNN(
    architecture=resnet,
    classes=class_list,
    sample_duration=clip_duration,  # 10s, selected above
    sample_rate=6000,
)

model2 = CNN(
    architecture=effnet,
    classes=class_list,
    sample_duration=clip_duration,  # 10s, selected above
    sample_rate=6000,
)


print(model1.device, model2.device)

cuda:0 cuda:0


Configure the wandb session

In [6]:
import wandb
from Keys import api_key


# try:
wandb.login()
wandb_session = wandb.init(
    entity="AI_for_Ecology_ITAW",  # replace with your entity/group name
    project="Airgun_detector",  # replace with your project name
    name=f"iteration {sequence} resnet",
)
# except:  # if wandb.init fails, don't use wandb logging
#     print("failed to create wandb session. wandb session will be None")
#     wandb_session = None

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/biofelip/.netrc.
wandb: Currently logged in as: felipe-1092 (AI_for_Ecology_ITAW) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
checkpoint_folder = step.joinpath("resnet")
checkpoint_folder.mkdir(exist_ok=True)

model1.train(train_df,
    val_df,
    steps=300,
    batch_size=128,
    log_interval=10,  # log progress every 100 batches
    validation_interval=5,
    # num_workers=num_workers,  # parallelized cpu tasks for preprocessing
    wandb_session=wandb_session,
    save_path=checkpoint_folder)  # location to save checkpoints)

wandb: WARNING Data passed to `wandb.Image` should consist of values in the range [0, 255], image data will be normalized to this range, but behavior will be removed in a future version of wandb.


  0%|          | 0/300 [00:00<?, ?it/s]

Step: 0/300 (0.0%)
	Running Average Loss: 0.809
	Most Recent Step Loss: 0.809

Validation.


  0%|          | 0/5 [00:00<?, ?it/s]


Validation.


  0%|          | 0/5 [00:00<?, ?it/s]

... (113 outputs left out)

  0%|          | 0/5 [00:00<?, ?it/s]


Validation.


  0%|          | 0/5 [00:00<?, ?it/s]


Best Model Appears at Step 255 with Validation score 0.969.


For the next experiment close the previous session in wandb and add a new one with a new name. 

In [8]:

wandb.finish()
wandb_session = wandb.init(
    entity="AI_for_Ecology_ITAW",  # replace with your entity/group name
    project="Airgun_detector",  # replace with your project name
    name=f"iteration {sequence} effnet",
)

loss,█▆▅▄▂▃▃▂▁▂▂▂▁▂▂▁▂▁▁▁▂▂▁▂▁▁▁▁▁▂
step,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
loss,0.25463
step,290


Train the effnet. 

In [9]:
effnet_folder = step.joinpath("effnet")
effnet_folder.mkdir(exist_ok=True)
model2.train(train_df,
    val_df,
    steps=300,
    batch_size=128,
    log_interval=10,  # log progress every 100 batches
    validation_interval=20,
    # num_workers=num_workers,  # parallelized cpu tasks for preprocessing
    wandb_session=wandb_session,
    save_path=effnet_folder)  # location to save checkpoints)

  0%|          | 0/300 [00:00<?, ?it/s]

Step: 0/300 (0.0%)
	Running Average Loss: 0.701
	Most Recent Step Loss: 0.701

Validation.


  0%|          | 0/5 [00:00<?, ?it/s]

Step: 10/300 (3.3%)
	Running Average Loss: 0.661
	Most Recent Step Loss: 0.630
Step: 20/300 (6.7%)
	Running Average Loss: 0.589
	Most Recent Step Loss: 0.542

Validation.


  0%|          | 0/5 [00:00<?, ?it/s]

Step: 30/300 (10.0%)
	Running Average Loss: 0.515
	Most Recent Step Loss: 0.461
Step: 40/300 (13.3%)
	Running Average Loss: 0.399
	Most Recent Step Loss: 0.418

Validation.


... (22 outputs left out)

  0%|          | 0/5 [00:00<?, ?it/s]

Step: 270/300 (90.0%)
	Running Average Loss: 0.228
	Most Recent Step Loss: 0.229
Step: 280/300 (93.3%)
	Running Average Loss: 0.249
	Most Recent Step Loss: 0.236

Validation.


  0%|          | 0/5 [00:00<?, ?it/s]

Step: 290/300 (96.7%)
	Running Average Loss: 0.228
	Most Recent Step Loss: 0.172

Best Model Appears at Step 220 with Validation score 0.959.


In [10]:
wandb.finish()

loss,█▇▆▅▄▄▂▃▁▁▃▂▂▃▂▃▁▃▂▂▁▂▂▂▁▂▁▂▂▁
step,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
loss,0.17169
step,290


Load the feature extractor. 

In [11]:
import bioacoustics_model_zoo as bmz
perch = bmz.Perch2ONNX(headless=True)
train_emb=np.asanyarray(perch.embed(train_df, return_dfs=False, return_preds=False))
val_emb=perch.embed(val_df, return_dfs=False, return_preds=False)

/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/ml/cnn.py:2930: UserWarning: The columns of input samples df differ from `model.classes`. Discarding sample df columns.
  warnings.warn(


  0%|          | 0/1779 [00:00<?, ?it/s]

/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/ml/cnn.py:2930: UserWarning: The columns of input samples df differ from `model.classes`. Discarding sample df columns.
  warnings.warn(


  0%|          | 0/630 [00:00<?, ?it/s]

Train the logistic regression.
- Iteratin 0 AUC  85%
- Iteration 1 AUC 92 %
- Iteration 2 AUC 91.9%
- Iteration 3 AUC 90.5%
- Iteration 4 AUC 92.86
- Iteration 5 AUC 92.11
- Iteration 6 AUC 93.49

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
# 3. Small classifier on top
clf = LogisticRegression(max_iter=5000, class_weight="balanced")   # balanced: you have few negatives
clf.fit(train_emb, train_df['airgun'])

val_scores = clf.predict_proba(val_emb)[:, 1]
print("val AUC:", roc_auc_score(val_df["airgun"], val_scores))


val AUC: 0.9349329057049062


Re train on the full dataset and save the shallow classifier

In [13]:
clf.fit(np.concatenate([train_emb, val_emb]), pd.concat([train_df, val_df])["airgun"])

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,5000
,multi_class,'deprecated'


In [14]:
import joblib
shallow_folder = step.joinpath("shallow_perch")
shallow_folder.mkdir(exist_ok=True)
joblib.dump(clf, shallow_folder.joinpath("shallow_classifier.joblib"))   # save

['annotation_step6/shallow_perch/shallow_classifier.joblib']

# Huber embeddings

- Iteration 3 AUC 80%
- Iteration 4 AUC 83%
- Iteration 5 AUC 86.6%
- Iteration 6 AUC 84.9%



In [15]:
import torch, torchaudio
from sklearn.linear_model import LogisticRegression

bundle = torchaudio.pipelines.HUBERT_BASE
hubert = bundle.get_model().eval().to("cuda")

def hubert_embed(file, start, end):
    x = Audio.from_file(file, offset=start, duration=end - start).resample(bundle.sample_rate).samples
    with torch.no_grad():
        feats, _ = hubert.extract_features(torch.tensor(x, dtype=torch.float32)[None].to("cuda"))
    return feats[-1].mean(dim=1).squeeze().cpu().numpy()      # last layer, averaged over time

E_train = np.array([hubert_embed(*i) for i in train_df.index])
E_val   = np.array([hubert_embed(*i) for i in val_df.index])

lr = LogisticRegression(max_iter=5000, class_weight="balanced").fit(E_train, train_df["airgun"])
print("val AUC:", roc_auc_score(val_df["airgun"], lr.predict_proba(E_val)[:, 1]))

/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/audio.py:2062: UserWarning: Audio object is shorter than requested duration: 2.0 sec instead of 5.0 sec
  warnings.warn(error_msg)
/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/audio.py:2062: UserWarning: Audio object is shorter than requested duration: 3.0 sec instead of 5.0 sec
  warnings.warn(error_msg)
/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/audio.py:2062: UserWarning: Audio object is shorter than requested duration: 2.0 sec instead of 5.0 sec
  warnings.warn(error_msg)
/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/audio.py:2062: UserWarning: Audio object is shorter than requested duration: 4.0 sec instead of 5.0 sec
  warnings.warn(error_msg)
/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/audio.py:2062: UserWarning: Audio object is shorter

val AUC: 0.8488525135777069


In [16]:
lr.fit(np.concatenate([E_train, E_val]), pd.concat([train_df, val_df])["airgun"])

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,5000
,multi_class,'deprecated'


In [17]:
waveform_folder = step.joinpath("shallow_HUbert")
waveform_folder.mkdir(exist_ok=True)
joblib.dump(lr, waveform_folder.joinpath("shallow_waveform_classifier.joblib"))   # save

['annotation_step6/shallow_HUbert/shallow_waveform_classifier.joblib']

In [20]:
import opensoundscape as opso
best_resnet = opso.load_model(f"annotation_step{sequence}/resnet/best.model")
best_effnet = opso.load_model(f"annotation_step{sequence}/effnet/best.model")

/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/preprocess/overlay.py:126: UserWarning: Overlay class's .overlay_df will be None after loading from dict and `.criterion_fn` will be always_true(). Reset these attributes and set .bypass to False to use Overlay after loading with from_dict().
  warnings.warn(
/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/preprocess/overlay.py:126: UserWarning: Overlay class's .overlay_df will be None after loading from dict and `.criterion_fn` will be always_true(). Reset these attributes and set .bypass to False to use Overlay after loading with from_dict().
  warnings.warn(


In [21]:
val_ens = pd.DataFrame({
    "resnet": best_resnet.predict(val_df, activation_layer="sigmoid", batch_size=32)["airgun"].values,
    "effnet": best_effnet.predict(val_df, activation_layer="sigmoid", batch_size=32)["airgun"].values,
    "perch":  val_scores,                                  # perch logistic regression
    "hubert": lr.predict_proba(E_val)[:, 1],
})
print("just resnet:", roc_auc_score(val_df["airgun"], val_ens["resnet"]))
print("just effnet:", roc_auc_score(val_df["airgun"], val_ens[ "effnet"]))
print("just perch:", roc_auc_score(val_df["airgun"], val_ens["perch"]))
print("resnet effent:", roc_auc_score(val_df["airgun"], val_ens[["resnet", "effnet"]].mean(axis=1)))
print("resnet perch:", roc_auc_score(val_df["airgun"], val_ens[["resnet", "perch"]].mean(axis=1)))
print("effent perch:", roc_auc_score(val_df["airgun"], val_ens[["effnet", "perch"]].mean(axis=1)))
print("3 models:", roc_auc_score(val_df["airgun"], val_ens[["resnet", "effnet", "perch"]].mean(axis=1)))
print("4 models:", roc_auc_score(val_df["airgun"], val_ens.mean(axis=1)))

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

just resnet: 0.9722243233839124
just effnet: 0.9608175367997458
just perch: 0.9349329057049062
resnet effent: 0.9736463896158908
resnet perch: 0.9795918367346939
effent perch: 0.9770200148257969
3 models: 0.9823603273778005
4 models: 0.9799246607464335


In [23]:
import numpy as np
from sklearn.metrics import roc_auc_score
y = val_df["airgun"].values
s3 = val_ens[["resnet", "effnet", "perch"]].mean(axis=1).values
s4 = val_ens[["resnet", "effnet", "perch", "hubert"]].mean(axis=1).values
s2 = val_ens[["resnet", "perch"]].mean(axis=1).values
s2_nl = val_ens[["resnet", "effnet"]].mean(axis=1).values

rng = np.random.default_rng(0)
diffs = []
for _ in range(2000):
    i = rng.integers(0, len(y), len(y))
    if y[i].min() != y[i].max():
        diffs.append(roc_auc_score(y[i], s2_nl[i]) - roc_auc_score(y[i], s3[i]))
print(np.percentile(diffs, [2.5, 97.5]))   # range of plausible differences between 3 and 2 models

[-0.01946746  0.00014126]
